# 05 — Demo Hybrid Risk Engine

Menyusun keluaran persis seperti contoh README §14: Risk Score, status,
Confidence Score, Data Quality Score, suspected area, indikator dominan,
dan rekomendasi pemeriksaan.

**Keluaran ini dihitung dari data sintetis** dan tidak boleh dijadikan
dasar tindakan operasi.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from backend.app.core.config import get_settings
settings = get_settings()
print("Akar proyek:", settings.paths.root)
print("Mode deployment:", settings.deployment_mode)

In [ ]:
from backend.app.reports.risk_demo import run

outcome = run(settings, model_name="xgboost", top=2)
print(outcome["snapshots"][0])

## Perjalanan skor menjelang event

In [ ]:
import matplotlib.pyplot as plt
from backend.app.reports import viz

viz.apply_theme()
scores = outcome["scores"].set_index("timestamp")
event_time = outcome["event_time"]

figure, axes = plt.subplots(figsize=(10, 5))
axes.plot(scores.index, scores["risk_score_persistent"], color=viz.CATEGORICAL[0],
          label="Risk Score")
axes.plot(scores.index, scores["confidence_score"], color=viz.CATEGORICAL[1],
          label="Confidence Score")
axes.plot(scores.index, scores["data_quality_score"], color=viz.CATEGORICAL[2],
          label="Data Quality Score")
axes.axvline(event_time, color=viz.STATUS["critical"], linewidth=2)
axes.annotate("event tercatat", xy=(event_time, 96), xytext=(8, 0),
              textcoords="offset points", color=viz.STATUS["critical"],
              fontsize=9.5, fontweight="600")

for band in settings.risk_bands:
    axes.axhline(band["min"], color=viz.GRIDLINE, linewidth=0.8, zorder=0)

viz.style_axes(
    axes,
    title="Tiga skor README §13 dilaporkan terpisah",
    subtitle="Risk Score tinggi di atas data buruk tidak berhak disebut meyakinkan.",
    xlabel="Waktu",
    ylabel="Skor 0-100",
)
axes.set_ylim(0, 105)
axes.legend(loc="lower left", ncols=3)
plt.tight_layout()

## Aturan yang menyala pada puncak risiko

In [ ]:
from backend.app.reports.risk_demo import load_artifacts, prepare_period
from backend.app.rules.risk_rules import rule_triggers_at, suspected_area

model, columns, threshold, baseline = load_artifacts(
    settings, "xgboost", settings.model["primary_horizon"]
)
features, raw = prepare_period(
    settings, baseline, scores.index[0], scores.index[-1]
)
position = int(outcome["scores"]["risk_score_persistent"].idxmax())

triggers = rule_triggers_at(features, position, settings)
print("Suspected area:", suspected_area(triggers, settings))
print()
for trigger in triggers:
    print(f"  [{trigger.points:>2.0f}] {trigger.rule:28} {trigger.describe()}")

## Berkas keluaran

In [ ]:
print(outcome["output_path"].read_text(encoding="utf-8")[:2000])